In [ ]:
import os
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"

DATA_PATH = "/content/drive/MyDrive/CheckThat-Task2"

if not os.path.exists(DATA_PATH):
    print("Downloading once to Drive...")
    !pip install gdown -q
    !gdown --folder "https://drive.google.com/drive/folders/1F7CuIwCuag3S9Qy4VqUyo18qLTqnxZDR"
    !gdown "https://drive.google.com/uc?id=11g-LyDVrMP09EimzKQIg0hXBvEXrKCQN"
    !mkdir -p "$DATA_PATH"
    !mv "/content/CheckThat-Task2/clef2026_gpt4_o_mini_train_arabic.json" "$DATA_PATH"
    !mv "/content/CheckThat-Task2/clef2026_gpt4_o_mini_val_english.json" "$DATA_PATH"
    !mv "/content/CheckThat-Task2/clef_2026_final_english_test.json" "$DATA_PATH"
else:
    print("Data already in Drive. Skipping download.")

In [ ]:
# Configuration
config = {
    "train_path": DATA_PATH + "/clef2026_gpt4_o_mini_train_arabic.json",  # Path to training data
    "val_path": DATA_PATH + "/clef2026_gpt4_o_mini_val_english.json",    # Path to validation data
    "test_path": DATA_PATH + "/clef_2026_final_english_test.json",
    "model_name": "google/gemma-4-E4B-it",
    "lora_rank": 16,
    "experiment_name": "clef-fact-check-verifier-gemma-4-e4b-lora-16",
    "is_sanity": True,
    "batch_size": 4,
    "epochs": 2,
    "lr": 2e-4, # Unsloth often works better with slightly higher LR
    "max_length": 2048,
    "output_dir": "/content/outputs"
}

In [ ]:
import os
import re
import json
import time
import datetime
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    get_cosine_schedule_with_warmup
)
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, classification_report
from huggingface_hub import HfApi, login

# Set default dtype
torch.set_default_dtype(torch.float32)

# Login to Hugging Face
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged into HuggingFace")
else:
    print("HF_TOKEN not found in environment. Please ensure you are logged in via CLI or set the variable.")

# Unified System Prompt for Gemma 4
SYSTEM_PROMPT = (
    "You are an expert fact-checking auditor. Your job is to evaluate whether a previous fact-checker's verdict and justification are correct given the claim and retrieved documents. "
    "You only respond with Yes or No, Yes if the Claim checker's Verdict and Justification are correct, No if they are incorrect."
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
def remove_label_pattern(text):
    text = re.sub(
        r"(\[?\s*Justification\s*\]?:?\s*)|(\[Label\]:\s*(True|False|Conflicting))",
        "",
        text, 
        flags=re.IGNORECASE
    ).strip()
    return text.replace("\n", " ")

def unroll_data(data):
    unrolled = []
    for idx, item in enumerate(data):
        label = item.get("label", "")
        claim = item.get("claim", "")
        verdict_list = item.get("Verdict_list", [])
        reasoning_traces = item.get("Reasoning_traces", [])
        sample_id = item.get("id", idx)
        
        for v, trace in zip(verdict_list, reasoning_traces):
            justification = remove_label_pattern(trace).split("Label:")[0].strip()
            class_label = 1 if str(v).lower() == str(label).lower() else 0
            
            unrolled.append({
                "sample_id": sample_id,
                "input_text": f"Claim: {claim}\nVerdict: {v}\nJustification: {justification}",
                "Label": label,
                "Verdict": v,
                "Class": class_label
            })
    return unrolled

# Load Data
print("Loading data...")
with open(config["train_path"], "r") as f: train_raw = json.load(f)
with open(config["val_path"], "r") as f: val_raw = json.load(f)
with open(config["test_path"], "r") as f: test_raw = json.load(f)

# Sanity Check Logic
if config["is_sanity"]:
    print("Sanity mode: using first 10 samples of each set.")
    train_raw = train_raw[:10]
    val_raw = val_raw[:10]
    test_raw = test_raw[:10]

train_unrolled = unroll_data(train_raw)
val_unrolled = unroll_data(val_raw)
test_unrolled = unroll_data(test_raw)

train_df = pd.DataFrame(train_unrolled)
val_df = pd.DataFrame(val_unrolled)
test_df = pd.DataFrame(test_unrolled)

print(f"Unrolled samples - Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

In [ ]:
class CustomClassifier(torch.nn.Module):
    def __init__(self, unsloth_model, tokenizer):
        super().__init__()
        self.model = unsloth_model
        # Gemma-4 usually has "Yes" and "No" in its vocab
        self.yes_token_id = tokenizer.convert_tokens_to_ids("Yes")
        self.no_token_id = tokenizer.convert_tokens_to_ids("No")
        
        print(f"Token IDs - Yes: {self.yes_token_id}, No: {self.no_token_id}")
        
        # If Yes/No not found, fallback to common Gemma tokens
        if self.yes_token_id == tokenizer.unk_token_id or self.yes_token_id is None:
            self.yes_token_id = tokenizer.encode("Yes", add_special_tokens=False)[0]
        if self.no_token_id == tokenizer.unk_token_id or self.no_token_id is None:
            self.no_token_id = tokenizer.encode("No", add_special_tokens=False)[0]

    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        # Take logits of the last token
        last_token_logits = outputs.logits[:, -1, :]
        # Index 0: No, Index 1: Yes
        target_logits = last_token_logits[:, [self.no_token_id, self.yes_token_id]]
        return target_logits

class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.texts = dataframe["input_text"].tolist()
        self.labels = dataframe["Class"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"{self.texts[idx]}\n\n[Evaluation]:"}
        ]
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        
        encoding = self.tokenizer(
            prompt, 
            truncation=True, 
            padding="max_length", 
            max_length=self.max_length, 
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [ ]:
class TrainerModule:
    def __init__(self, model, train_loader, val_loader, tokenizer, config):
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)
        self.tokenizer = tokenizer
        self.train_loader = train_loader
        self.val_loader = val_loader
        
        self.optimizer = AdamW(model.parameters(), lr=config["lr"], eps=1e-8)
        self.loss_fn = torch.nn.CrossEntropyLoss()
        
        total_steps = len(train_loader) * config["epochs"]
        self.scheduler = get_cosine_schedule_with_warmup(
            self.optimizer, 
            int(0.05 * total_steps), 
            total_steps
        )
        
        self.api = HfApi()
        self.exp_name = config["experiment_name"]
        self.output_dir = os.path.join(config["output_dir"], self.exp_name)
        os.makedirs(self.output_dir, exist_ok=True)

    def upload_to_hf(self, epoch):
        try:
            repo_id = f"{self.exp_name}-lora"
            print(f"Uploading checkpoint for epoch {epoch} to HuggingFace...")
            # Unsloth models are saved via the underlying PEFT model
            self.model.model.model.save_pretrained(self.output_dir)
            self.tokenizer.save_pretrained(self.output_dir)
            self.api.upload_folder(
                folder_path=self.output_dir,
                repo_id=repo_id,
                repo_type="model",
                commit_message=f"Epoch {epoch} checkpoint"
            )
            print("Upload successful.")
        except Exception as e:
            print(f"Failed to upload to HF: {e}")

    def train(self):
        for epoch in range(self.config["epochs"]):
            print(f"\nEpoch {epoch+1}/{self.config['epochs']}")
            self.model.train()
            total_loss, total_acc = 0, 0
            
            for batch in tqdm(self.train_loader, desc="Training"):
                self.optimizer.zero_grad()
                ids, mask, labels = batch["input_ids"].to(self.device), batch["attention_mask"].to(self.device), batch["labels"].to(self.device)
                logits = self.model(ids, mask)
                loss = self.loss_fn(logits, labels)
                loss.backward()
                self.optimizer.step()
                self.scheduler.step()
                total_loss += loss.item()
                preds = torch.argmax(logits, dim=-1).cpu().numpy()
                total_acc += accuracy_score(labels.cpu().numpy(), preds)
            
            print(f"Train Loss: {total_loss/len(self.train_loader):.4f} | Train Acc: {total_acc/len(self.train_loader):.4f}")
            self.evaluate(epoch)
            self.upload_to_hf(epoch)

    def evaluate(self, epoch):
        self.model.eval()
        total_loss, total_acc = 0, 0
        with torch.no_grad():
            for batch in tqdm(self.val_loader, desc="Validating"):
                ids, mask, labels = batch["input_ids"].to(self.device), batch["attention_mask"].to(self.device), batch["labels"].to(self.device)
                logits = self.model(ids, mask)
                loss = self.loss_fn(logits, labels)
                total_loss += loss.item()
                preds = torch.argmax(logits, dim=-1).cpu().numpy()
                total_acc += accuracy_score(labels.cpu().numpy(), preds)
        print(f"Val Loss: {total_loss/len(self.val_loader):.4f} | Val Acc: {total_acc/len(self.val_loader):.4f}")

In [ ]:
# Initialize Unsloth Model and Tokenizer
model, tokenizer = FastModel.from_pretrained(
    model_name = config["model_name"],
    max_seq_length = config["max_length"],
    dtype = None,
    load_in_4bit = True,
)

# Apply Gemma 4 Chat Template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4",
)

# Setup PEFT for Gemma 4 (Multimodal aware)
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Text-only SFT
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = config["lora_rank"],
    lora_alpha = config["lora_rank"] * 2,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

classifier = CustomClassifier(model, tokenizer)

train_dataset = TextDataset(train_df, tokenizer, config["max_length"])
val_dataset = TextDataset(val_df, tokenizer, config["max_length"])

train_loader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config["batch_size"])

trainer = TrainerModule(classifier, train_loader, val_loader, tokenizer, config)
trainer.train()

In [ ]:
class VerifierEvaluator:
    def __init__(self, classifier, tokenizer, device):
        self.classifier = classifier
        self.tokenizer = tokenizer
        self.device = device
        self.classifier.eval()

    def score_batch(self, claims, verdicts, justifications, max_length=2048):
        texts = []
        for c, v, j in zip(claims, verdicts, justifications):
            input_content = f"Claim: {c}\nVerdict: {v}\nJustification: {j}\n\n[Evaluation]:"
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": input_content}
            ]
            text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            texts.append(text)

        encoding = self.tokenizer(
            texts, 
            truncation=True, 
            padding="max_length", 
            max_length=max_length, 
            return_tensors="pt"
        )
        ids = encoding["input_ids"].to(self.device)
        mask = encoding["attention_mask"].to(self.device)
        with torch.no_grad():
            logits = self.classifier(ids, mask)
            return logits[:, 1].tolist()

def run_full_evaluation(evaluator, raw_data, batch_size=8):
    flat_inputs = []
    flat_indices = []
    for idx, sample in enumerate(raw_data):
        for t_idx, trace in enumerate(sample["Reasoning_traces"]):
            justification = remove_label_pattern(trace).split("Label:")[0].strip()
            flat_inputs.append({
                "claim": sample["claim"],
                "verdict": sample["Verdict_list"][t_idx],
                "justification": justification
            })
            flat_indices.append((idx, t_idx))
    all_scores = []
    for i in tqdm(range(0, len(flat_inputs), batch_size), desc="Full Evaluation"):
        batch = flat_inputs[i:i+batch_size]
        scores = evaluator.score_batch(
            [b["claim"] for b in batch],
            [b["verdict"] for b in batch],
            [b["justification"] for b in batch]
        )
        all_scores.extend(scores)
    results = [{"scores": []} for _ in range(len(raw_data))]
    for score, (sample_idx, _) in zip(all_scores, flat_indices):
        results[sample_idx]["scores"].append(score)
    predictions = []
    for idx, sample in enumerate(raw_data):
        s_list = results[idx]["scores"]
        best_trace_idx = np.argmax(s_list)
        best_verdict = sample["Verdict_list"][best_trace_idx]
        predictions.append({
            "query_id": idx,
            "Claim": sample["claim"],
            "Label": sample["label"],
            "Verdict_BoN": best_verdict,
            "BoN_Verdict_list": sample["Verdict_list"],
            "scores": s_list
        })
    return predictions

evaluator = VerifierEvaluator(classifier, tokenizer, device)
test_predictions = run_full_evaluation(evaluator, test_raw)
with open(os.path.join(trainer.output_dir, "test_predictions.json"), "w") as f:
    json.dump(test_predictions, f, indent=4)
print("Evaluation complete. Results saved.")

In [ ]:
# Final Metrics
y_true = [p["Label"].lower() for p in test_predictions]
y_pred = [p["Verdict_BoN"].lower() for p in test_predictions]
print("\nClassification Report:")
print(classification_report(y_true, y_pred, zero_division=0))
def calculate_recall_at_k(preds, k):
    recalls = []
    for p in preds:
        label = p["Label"].lower()
        verdicts = [v.lower() for v in p["BoN_Verdict_list"]]
        scores = p["scores"]
        ranked_indices = np.argsort(scores)[::-1]
        top_k_indices = ranked_indices[:k]
        total_relevant = sum(1 for v in verdicts if v == label)
        if total_relevant == 0: continue
        retrieved_relevant = sum(1 for i in top_k_indices if verdicts[i] == label)
        recalls.append(retrieved_relevant / total_relevant)
    return np.mean(recalls) if recalls else 0.0
for k in [1, 3, 5]:
    r_k = calculate_recall_at_k(test_predictions, k)
    print(f"Recall@{k}: {r_k:.4f}")